# Cooking Recommendation Chatbot

 the user describes what they're in the mood to eat — a craving ("something light for dinner"), an intensity ("something strong, I'm starving"), or a cuisine ("Mexican food") — and the app generates a complete suggestion: a dish, why it fits, a vivid description, ingredients, step-by-step preparation, a short history of the dish, and illustrative example reviews with star ratings, plus a generated image of the finished dish.


**Models / APIs used :**
- [Groq](https://console.groq.com) — Prompt Guard 2 (security check), GPT-OSS 20B(topic check) and GPT-OSS 120B (main generation)
- [Pollinations.ai](https://pollinations.ai) — free image generation, no key needed (throttled to about 1 image per 15 seconds). If it fails, the app shows the dishname instead.


## 0. Setup

In [1]:
# Install the Groq Python client, Gradio, and Pillow (image handling).
!pip install -q groq gradio pillow requests


In [2]:
from getpass import getpass
import os

#get the api key
# getpass hides the input so it doesn't get printed or saved in the notebook.
os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API key: ")



Enter your Groq API key: ··········


In [3]:
import io
import json
import os
import random
import re
import time

import groq
import gradio as gr
import requests
from groq import Groq
from PIL import Image
client = Groq(api_key=os.environ["GROQ_API_KEY"], timeout=45.0, max_retries=2)




PROMPT_GUARD_MODEL = "meta-llama/llama-prompt-guard-2-86m"  # prompt guard (classifier ): for injection / jailbreak detection
MAIN_MODEL = "openai/gpt-oss-120b"                          # main content generation
RELEVANCE_MODEL = "openai/gpt-oss-20b"                      # topic check: yes/no (food related)



# Parameters
MAIN_TEMPERATURE = 0.8   # high temperature, more varied/creative dish suggestions and reviews

MAIN_TOP_P = 0.95        # remove the very unlikely words to keep output coherent (the higher top_p the wider set of tokens will be considered/accepted)

MAIN_MAX_TOKENS = 2500   # gpt-oss-120b max tokens for the recipe generation, ingredients, reviews ...
                         # Not higher: bc groq's free tier allows only 30 requests/min, 8K tokens/minute for this model, and the overall used tokens is close to that limit




MAIN_REASONING_EFFORT = "low"  # keeps reasoning small, no need for complex thinking

RETRY_TEMPERATURE = 0.4  # if the first answer is malformed, retry with a lower temperature




RATING_TARGET_RANGE = (2.8, 4.9)  # food rating range



# Input / safety parameters
MAX_INPUT_CHARS = 300            # reject huge inputs from the user, anything more than 300 characters
PROMPT_GUARD_THRESHOLD = 0.5     # risk score above this = flagged (ATTACK)
DEBUG = True                     # see the results of the prompt guard (risk level) and topic check (related or not)


IMAGE_TIMEOUT = 30 #image parameter


## 1. Error handling: API + the two input checks (Persona + few shots)


1. **Security check (Prompt Guard)** -- catches prompt injection / jailbreak attempts.
2. **Topic relevance check (Persona + few-shot classification)** -- a separate call that answers YES/NO to "is this about food?", with a some examples in its prompt.


In [4]:
def friendly_error(e): #API error handling
    """Turns API exceptions into short messages a user can understand."""
    if isinstance(e, groq.AuthenticationError):
        return "The Groq API key was rejected. Please re-run the API key cell with a valid key."
    if isinstance(e, groq.RateLimitError):
        return "The free API rate limit was reached. Please wait a minute and try again."
    if isinstance(e, groq.APITimeoutError):  # must come before APIConnectionError (it is a subclass)
        return "The AI service took too long to respond. Please try again."
    if isinstance(e, groq.APIConnectionError):
        return "Could not reach the AI service. Please check your connection and try again."
    return f"Unexpected error: {e}"


def check_input_safety(user_text): # injection detection , returns True if the input is safe otherwise returns false but a sentence appears in the interface (flagged)
    try:
        response = client.chat.completions.create(
            model=PROMPT_GUARD_MODEL, #llama model gives a score between 0 and 1, if it’s above 0.5 then it’s unsafe.
            messages=[{"role": "user", "content": user_text}],
        )
        raw_output = (response.choices[0].message.content or "").strip()
        if DEBUG:
            print(f"[debug] Prompt Guard raw output: {raw_output!r}")


        try:
            is_flagged = float(raw_output) > PROMPT_GUARD_THRESHOLD #0.5
        except ValueError:
            upper = raw_output.upper()
            is_flagged = any(word in upper for word in ["MALICIOUS", "INJECTION", "JAILBREAK"])

        return not is_flagged
    except Exception as e:
        print(f"[Prompt Guard error, continuing cautiously]: {friendly_error(e)}")
        return True


# the use of PERSONA and FEW-SHOTS for the topic check classification (gpt 20b)
RELEVANCE_PROMPT = """You are a strict classifier. Reply with exactly one word: YES or NO.
YES = the message is about food, cooking, recipes, hunger, or a craving / meal / cuisine preference.
NO = anything else (vehicles, sports, people, news, geography, coding, and so on).

Examples:
"something spicy for dinner" -> YES
"Italian food" -> YES
"I'm starving" -> YES
"a flying car" -> NO
"a man driving" -> NO
"highest mountains in the world" -> NO"""


def check_topic_relevance(user_text): #returns True is the input is food related otherwise returns false but a sentence appears in the interface(i'm food assitant) + pic of a chef
    try:
        response = client.chat.completions.create(
            model=RELEVANCE_MODEL, #gpt 20b
            messages=[
                {"role": "system", "content": RELEVANCE_PROMPT},
                {"role": "user", "content": user_text},
            ],
            temperature=0,   # deterministic: a classifier should not be "creative"
            max_tokens=250,
            reasoning_effort="low",
        )
        answer = (response.choices[0].message.content or "").strip().upper()
        if DEBUG:
            print(f"[debug] Relevance check answer: {answer!r}")
        if not answer:
            print("[Relevance check returned nothing, continuing cautiously]")
            return True
        return not answer.startswith("N")
    except Exception as e:
        print(f"[Relevance check error, continuing cautiously]: {friendly_error(e)}")
        return True


## 2. Main generation call

This is where the recipe content is created. The prompt combines these techniques:

- **Persona** : the model is "Chef Aria".
- **One-shot** : a complete example of the exact JSON shape expected.
- **Chain-of-thought** : in `reasoning` field, the model has to think and explain to the user why it thinks that dish fits him (cuisine, effort needed, desired meal heaviness)
- **Constrained output format** : strict JSON plus explicit rules (exactly 3 reviews, integer ratings).
- **Model-written image prompt** : the JSON has an `image_prompt` field, so the model itself writes a detailed photo description (plating, colors, garnish, surface, lighting, camera angle) that is sent to the image service together with a fixed style line.
- **Controlled variation** :the program picks a random target average (2.8-4.8) for each request and passes it to the model, so the star rating varies


In [5]:
SYSTEM_PROMPT = """You are Chef Aria, a friendly, knowledgeable cooking assistant with a warm, encouraging tone.
A user will describe a craving, mood, intensity, or cuisine preference
(for example "something light", "something strong, I'm starving", or "Mexican food"). Your job:

1. Think about which single dish best fits the request: how light or heavy it should be, the cuisine,
   and the effort involved.
   Write this thinking as the FIRST field, "reasoning", BEFORE you choose the dish.
   Speak directly and warmly to the person using "you" (for example "Since you're after something light, ...").
   Never write "the user" in any field.

2. Then respond with ONLY a single valid JSON object, no extra text before or after it, in exactly this shape:

{
  "reasoning": "1-2 warm sentences, addressed to 'you', on why this dish fits what they asked for",
  "dish_name": "string",
  "description": "2-3 vivid sentences about the dish itself: how it looks, smells and tastes, its textures and main flavors. Do not mention the request.",
  "image_prompt": "ONE English sentence of 40-70 words describing how the finished dish looks in a photograph. Start with the main food item and its most recognizable visible features (for example bone-in ribs with the bone showing, a sliced steak with a pink center, melted cheese). Then give the plating on a LIGHT-colored plate or bowl, the garnish and side items, a contrasting table surface, bright even daylight and the camera angle. The food must be clearly recognizable, not a dark blur. Describe only what is visible. No people, no text, no brand names.",
  "ingredients": ["ingredient with quantity", "..."],
  "steps": ["step 1", "step 2", "..."],
  "history": "2-4 sentence background/history of the dish",
  "reviews": [
    {"reviewer": "a plausible first name and country", "rating": 1-5, "text": "short review text"},
    {"reviewer": "...", "rating": 1-5, "text": "..."},
    {"reviewer": "...", "rating": 1-5, "text": "..."}
  ]
}

Rules:
- Give 5-10 ingredients (with quantities) and 4-8 clear preparation steps.
- Give exactly 3 reviews with whole-number ratings from 1 to 5. The user message states a target average rating:
  make the three ratings average roughly that value, with real spread between them, and make each review's text
  match its rating (a 2-star review mentions a genuine downside such as too salty, too much work, or too bland).
- Only output the JSON object, nothing else.

  Example (for a request like "something light for lunch", target average rating 3.3):
{
  "reasoning": "Since you're after something light, a fresh no-cook vegetable salad fits better than anything heavy or fried.",
  "dish_name": "Greek Salad",
  "description": "Juicy tomato wedges, crisp cucumber and sharp red onion are tossed with briny kalamata olives and topped with a creamy slab of feta. Everything is glossy with fruity olive oil and fragrant with dried oregano, so every bite is cool, salty and bright.",
  "image_prompt": "A colorful Greek salad of ripe red tomato wedges, green cucumber slices, purple red onion rings and dark kalamata olives, topped with a thick slab of white feta dusted with oregano and drizzled with golden olive oil, served in a white ceramic bowl on a weathered wooden table, bright natural daylight, shot from a 45-degree angle",
  "ingredients": ["4 ripe tomatoes", "1 cucumber", "1/2 red onion", "1/2 cup kalamata olives", "150 g feta cheese", "3 tbsp olive oil", "1 tsp dried oregano"],
  "steps": ["Chop the tomatoes, cucumber, and red onion into chunks.", "Combine in a bowl with the olives.", "Top with a slab of feta.", "Drizzle with olive oil and sprinkle with oregano."],
  "history": "Greek salad, or horiatiki, originated in rural Greece as a simple farmer's dish using whatever vegetables were on hand.",
  "reviews": [
    {"reviewer": "Maria, Greece", "rating": 5, "text": "Tastes just like my grandmother's version!"},
    {"reviewer": "Tom, USA", "rating": 3, "text": "Fresh, but a bit plain without some bread on the side."},
    {"reviewer": "Priya, India", "rating": 2, "text": "Way too salty for me, the feta and olives overpowered everything."}
  ]
}"""

REQUIRED_FIELDS = {
    "dish_name": str, "description": str, "ingredients": list,
    "steps": list, "history": str, "reviews": list,
}


def parse_json_response(raw_text):
    """Turns the model's text into a Python dict. Tolerates markdown code fences
    and stray text around the JSON object."""
    text = (raw_text or "").strip()
    text = re.sub(r"^```(?:json)?|```$", "", text, flags=re.MULTILINE).strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        start, end = text.find("{"), text.rfind("}")
        if start != -1 and end > start:
            return json.loads(text[start:end + 1])
        raise


def validate_recipe(recipe): #missing field error handling
    """Raises ValueError if the parsed object is missing required fields."""
    if not isinstance(recipe, dict):
        raise ValueError("the response was not a JSON object")
    bad = [k for k, t in REQUIRED_FIELDS.items() if not isinstance(recipe.get(k), t) or not recipe.get(k)]
    if bad:
        raise ValueError(f"missing or empty fields: {bad}")


def generate_recipe(user_text, retries=1): #the recipe/dish generator
    """Calls the main model to generate the structured recipe package.
    Returns (recipe_dict, None) on success or (None, error_message) on failure.
    Malformed output is retried (at a lower temperature); API errors return immediately
    with a friendly message because the client already retried timeouts / rate limits."""
    last_error = None
    use_json_mode = True
    target_rating = round(random.uniform(*RATING_TARGET_RANGE), 1)   # varies the star rating per request so it gives different ratings, so with each request it chooses a number in the rating_target_range(above) and send it to the model to generate ratings which average is close to that number
    for attempt in range(retries + 1):
        temperature = MAIN_TEMPERATURE if attempt == 0 else RETRY_TEMPERATURE
        try:
            kwargs = dict(
                model=MAIN_MODEL,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": f"User request: {user_text}\n\nTarget average rating for the example reviews: {target_rating} / 5"},
                ],
                temperature=temperature,
                top_p=MAIN_TOP_P,
                max_tokens=MAIN_MAX_TOKENS,
                reasoning_effort=MAIN_REASONING_EFFORT,
            )
            if use_json_mode:
                kwargs["response_format"] = {"type": "json_object"}
            response = client.chat.completions.create(**kwargs)

            recipe = parse_json_response(response.choices[0].message.content)
            validate_recipe(recipe)
            return recipe, None
        except (json.JSONDecodeError, ValueError) as e:
            last_error = f"The AI's response was malformed (attempt {attempt + 1}): {e}"
        except groq.BadRequestError as e:
            use_json_mode = False
            last_error = f"Request rejected (attempt {attempt + 1}): {e}"
        except Exception as e:
            return None, friendly_error(e)
        time.sleep(1)
    return None, last_error


## 3. Processing the generated data

Deterministic work done in my own code on top of what the AI produced -- validating ratings, computing the average, and formatting everything for display.

In [6]:
def _safe_rating(review): #rating error handling, eg the model return invalid rating like 7/5
    """Returns the review's rating as an int clamped to 1-5, or None if it can't be read."""
    try:
        return min(5, max(1, int(round(float(review.get("rating"))))))
    except (TypeError, ValueError):
        return None


def process_recipe(recipe): #computes the actual average of the ratings
    """Computes the average rating and formats ingredients/steps/reviews as markdown."""
    reviews = [r for r in recipe.get("reviews", []) if isinstance(r, dict)]
    ratings = [x for x in (_safe_rating(r) for r in reviews) if x is not None]
    avg_rating = sum(ratings) / len(ratings) if ratings else None

    ingredients_md = "\n".join(f"- {i}" for i in recipe.get("ingredients", []))
    steps_md = "\n".join(f"{idx + 1}. {s}" for idx, s in enumerate(recipe.get("steps", [])))

    review_blocks = []
    for r in reviews:
        stars = "⭐" * (_safe_rating(r) or 0)
        review_blocks.append(f"**{r.get('reviewer', 'Anonymous')}** — {stars}\n{r.get('text', '')}")
    reviews_md = "\n\n".join(review_blocks)

    return avg_rating, len(ratings), ingredients_md, steps_md, reviews_md


## 4. Image generation (Pollinations.ai)

Groq has no image models, so the dish picture comes from Pollinations, a separate free service that needs no API key. It allows about one image every 15 seconds per IP, so the code spaces its requests and waits/retries when throttled (an image can take up to ~30 s). If it still fails, the app draws a simple placeholder card with Pillow so the UI never shows an empty box. For off-topic requests the picture slot shows a smiling cartoon chef that the code draws itself with Pillow (no image service involved).

In [7]:
def make_placeholder_image(dish_name): #Draws a simple placeholder card (used when the AI image is unavailable), contains the dish name

    from PIL import ImageDraw
    img = Image.new("RGB", (512, 512), (255, 236, 210))
    draw = ImageDraw.Draw(img)
    draw.rectangle([16, 16, 495, 495], outline=(214, 120, 60), width=6)
    text = dish_name if len(dish_name) <= 28 else dish_name[:25] + "..."
    draw.text((256, 250), text, fill=(120, 60, 20), anchor="mm")
    draw.text((256, 290), "(image unavailable)", fill=(160, 110, 70), anchor="mm")
    return img


def make_chef_image(): #Draws a smiling chef, appears when the user prompt is not food related for example
    """Draws a smiling chef (yellow emoji-style face, chef hat and coat, holding a spatula and a whisk).
    Drawn by the code itself with Pillow -- no AI image service involved. Shown for off-topic requests."""
    from PIL import ImageDraw
    S = 1024
    img = Image.new("RGB", (S, S), (255, 236, 210))
    d = ImageDraw.Draw(img)
    edge = (150, 150, 160)
    yellow, yellow_edge = (255, 204, 64), (214, 160, 30)

    # chef coat with buttons
    d.rounded_rectangle((232, 640, 792, 1100), radius=140, fill="white", outline=edge, width=8)
    for x in (440, 584):
        for y in (790, 870, 950):
            d.ellipse((x - 16, y - 16, x + 16, y + 16), fill=(190, 190, 200))

    # sleeves (outline pass first, then white pass, so the two arms look like solid shapes)
    arms = [((270, 720), (195, 560)), ((754, 720), (829, 560))]
    for width, color in ((108, edge), (92, "white")):
        for (x0, y0), (x1, y1) in arms:
            d.line((x0, y0, x1, y1), fill=color, width=width)
            for cx, cy in ((x0, y0), (x1, y1)):
                d.ellipse((cx - width // 2, cy - width // 2, cx + width // 2, cy + width // 2), fill=color)

    # red neckerchief, then the head
    d.polygon([(425, 650), (599, 650), (512, 770)], fill=(214, 60, 60))
    d.ellipse((322, 300, 702, 680), fill=yellow, outline=yellow_edge, width=8)

    # face: eyes, rosy cheeks, big smile
    for cx in (462, 562):
        d.ellipse((cx - 17, 430, cx + 17, 492), fill=(60, 40, 20))
        d.ellipse((cx - 8, 440, cx + 6, 456), fill="white")
    for cx in (402, 622):
        d.ellipse((cx - 28, 500, cx + 28, 548), fill=(255, 150, 130))
    d.arc((432, 450, 592, 610), start=25, end=155, fill=(120, 60, 20), width=16)

    # chef hat: puffs (outline pass, then white pass) + band
    puffs = [(405, 215, 88), (619, 215, 88), (512, 165, 105)]
    for cx, cy, r in puffs:
        d.ellipse((cx - r - 5, cy - r - 5, cx + r + 5, cy + r + 5), fill=edge)
    d.rounded_rectangle((375, 245, 649, 340), radius=26, fill=edge)
    for cx, cy, r in puffs:
        d.ellipse((cx - r, cy - r, cx + r, cy + r), fill="white")
    d.rounded_rectangle((380, 250, 644, 335), radius=24, fill="white")

    # tools: spatula (left hand) and whisk (right hand); handles first, then hands on top
    brown, metal, metal_edge = (140, 90, 50), (175, 180, 190), (110, 115, 125)
    d.line((195, 560, 195, 380), fill=brown, width=26)
    d.line((829, 560, 829, 440), fill=brown, width=26)
    for hx, hy in ((195, 560), (829, 560)):
        d.ellipse((hx - 52, hy - 52, hx + 52, hy + 52), fill=yellow, outline=yellow_edge, width=8)
    d.rounded_rectangle((140, 230, 250, 380), radius=22, fill=metal, outline=metal_edge, width=8)
    for x in (172, 195, 218):
        d.line((x, 255, x, 335), fill=metal_edge, width=8)
    for half in (46, 28, 10):
        d.ellipse((829 - half, 250, 829 + half, 440), outline=metal_edge, width=7)
    d.rectangle((813, 428, 845, 452), fill=metal_edge)

    return img.resize((512, 512), Image.LANCZOS)


FREE_POLLINATIONS_GAP = 16   # 16 seconds delay between requests (limit is about 1 per 15 s per IP), so after that the image request FINISHES, the prog needs to wait at least 16 sec before next request (to avoid pollinations payement error)
_last_free_call = [0.0]


def _image_from_pollinations_free(prompt):
    """Keyless Pollinations endpoint. Throttled to about 1 request / 15 s, so we wait if needed."""
    wait = FREE_POLLINATIONS_GAP - (time.time() - _last_free_call[0])
    if wait > 0:
        time.sleep(wait)
    _last_free_call[0] = time.time()
    url = f"https://image.pollinations.ai/prompt/{requests.utils.quote(prompt)}?width=512&height=512&nologo=true"
    try:
        response = requests.get(url, timeout=60, headers={"User-Agent": "Mozilla/5.0 (cooking-app)"})
    finally:
        _last_free_call[0] = time.time()   # the 15 s window is counted from when the last request FINISHED
    response.raise_for_status()
    if not response.headers.get("Content-Type", "").startswith("image"):
        raise ValueError(f"unexpected content type: {response.headers.get('Content-Type')}")
    return Image.open(io.BytesIO(response.content)).convert("RGB")


# Fixed photo-style words added after the model-written image_prompt. Edit this line to change the look of ALL images.
IMAGE_STYLE = ("professional food photography, 85mm lens, shallow depth of field, bright natural daylight, "
               "appetizing, vibrant colors, high detail, sharp focus, no text, no watermark")


def generate_dish_image(dish_name, image_prompt):
    """Tries each image provider in order.
    Returns (PIL image, True) on success, or (placeholder image, False) if none works."""
    prompt = f"{image_prompt}, {IMAGE_STYLE}"
    print(f"[debug] Image prompt: {prompt}")

    providers = []   # (name, function, max attempts)
    providers.append(("Pollinations (no key)", _image_from_pollinations_free, 3)) #max attempts is 3, so the prog will call pollitnations up to 3 times for the same request until it generates otherwise it will show the placeholder

    for name, fetch, attempts in providers:
        for attempt in range(attempts):
            try:
                return fetch(prompt), True
            except requests.HTTPError as e:
                code = e.response.status_code if e.response is not None else None
                print(f"[Image error from {name}, attempt {attempt + 1}]: {e}")
                if fetch is _image_from_pollinations_free and code in (402, 429):
                    continue   # throttled: the next attempt waits for the 15 s window automatically
                if code in (400, 401, 402, 403, 429):
                    break      # key / budget / daily-limit problem: try the next provider
                time.sleep(1)
            except Exception as e:
                print(f"[Image error from {name}, attempt {attempt + 1}]: {e}")
                time.sleep(1)
    return make_placeholder_image(dish_name), False


## 5. Putting it all together

The single function Gradio calls on each request. Pipeline: **input validation → Prompt Guard → topic check → recipe generation → processing → image generation.** Everything is wrapped so an unexpected error shows a message instead of crashing the app.

In [13]:
OFF_TOPIC_MESSAGES = [ #one of them appears when the prompt is not food related
    "I'd love to help with that, but I'm just a food advisor! 🍽️",
    "That's a bit outside my kitchen! 👩‍🍳 I'm a food advisor specialist -- let me help you find something delicious instead.",
    "Hmm, I only speak fluent food! 🍕 Give me a craving, a mood or a cuisine and I'll cook something up.",
    "Sorry, my expertise stops at the dinner table! 🥘 Ask me what to eat and I'll shine.",
]


def _fail(message, image=None):
    """Return value for the UI when the pipeline stops early (image + 5 text panels)."""
    return image, message, "", "", "", ""


def _run_pipeline(user_text):
    # Stage 0: basic input validation (empty / too long)
    if not user_text or not user_text.strip():
        return _fail("I'm ready to assist you in choosing your meal! 🍽️", make_chef_image())
    user_text = user_text.strip()
    if len(user_text) > MAX_INPUT_CHARS:
        return _fail(f"That's a bit long -- please keep it under {MAX_INPUT_CHARS} characters.")

    # Stage 1: security check (injection / jailbreak)
    if not check_input_safety(user_text):
        return _fail("That request was flagged as unsafe, so I can't process it. Please rephrase.")

    # Stage 2: topic relevance check
    if not check_topic_relevance(user_text):
        return _fail(random.choice(OFF_TOPIC_MESSAGES), make_chef_image())   # smiling chef drawn by our own code

    # Stage 3: main generation
    recipe, error = generate_recipe(user_text)
    if recipe is None:
        return _fail(f"Sorry, I couldn't generate your recipe.\n\n{error}")

    # Stage 4: processing (rating)
    avg_rating, n_ratings, ingredients_md, steps_md, reviews_md = process_recipe(recipe)
    rating_line = f"⭐ {avg_rating:.1f} / 5 (from {n_ratings} example reviews)" if avg_rating else ""

    # Stage 5: image generation (non-blocking -- the app still works if this fails)
    # The main model writes a dedicated, detailed image prompt; fall back to name + description if it is missing.
    image_prompt = recipe.get("image_prompt") or f"{recipe['dish_name']}, {recipe['description'][:200]}"
    image, image_is_real = generate_dish_image(recipe["dish_name"], image_prompt)
    image_note = "" if image_is_real else ""

    why = f"\n\n💡 **Why I picked it:** {recipe['reasoning']}" if recipe.get("reasoning") else ""
    description_md = f"## {recipe['dish_name']}\n{recipe['description']}\n\n{rating_line}{why}{image_note}"
    ingredients_md = "### 🧂 Ingredients\n" + ingredients_md
    steps_md = "### 👩‍🍳 Preparation\n" + steps_md
    history_md = "### 📜 History\n"+ recipe["history"]
    reviews_out = "### 💬 Example reviews\n" + reviews_md

    return image, description_md, ingredients_md, steps_md, history_md, reviews_out


def cooking_assistant(user_text):
    try:
        return _run_pipeline(user_text)
    except Exception as e:  # last line of defence: never let the UI crash
        print(f"[Unexpected pipeline error]: {e}")
        return _fail(f"Something unexpected went wrong. Please try again.\n\n({e})")


## 6. Gradio interface

In [ ]:
with gr.Blocks(title="Cooking Recommendation Chatbot") as demo:
    gr.Markdown("# 🍳 What should I cook?")
    gr.Markdown("Describe a craving, a mood, or a cuisine -- e.g. *\"something light for dinner\"* or *\"Mexican food\"*.")

    with gr.Row():
        user_input = gr.Textbox(label="What are you in the mood for?", placeholder="e.g. something spicy for dinner",
                                max_lines=2, scale=4)
        submit_btn = gr.Button("Get a recommendation", variant="primary", scale=1)

    status_output = gr.Markdown()   # shows the "please wait" sentence while a request is running

    gr.Examples(
        examples=["something light for dinner", "something strong, I'm starving", "Mexican food", "a cozy dessert for a rainy day"],
        inputs=user_input,
    )

    with gr.Row():
        with gr.Column(scale=1):
            image_output = gr.Image(label="Dish image", type="pil")
        with gr.Column(scale=2):
            description_output = gr.Markdown()

    with gr.Row():
        ingredients_output = gr.Markdown()
        steps_output = gr.Markdown()

    history_output = gr.Markdown()
    reviews_output = gr.Markdown()

    outputs = [image_output, description_output, ingredients_output, steps_output, history_output, reviews_output]

    def lock_ui(): # locks the input box when a request is already active to prevent spamming and loss of tokens (gpt 120b limits: 30 requests/min, 8k tokens/min)
        """Disables the button and the text box while a request runs, and shows the waiting sentence.
        A second submit can't start until the first one has finished, so no duplicate requests are sent
        (a new request is only possible once the user retypes it after the result appears)."""
        return (gr.update(interactive=False), gr.update(interactive=False),
                "⏳ Cooking up your recipe... generating the image can take up to 30 seconds, so please wait a bit.")

    def unlock_ui():
        return gr.update(interactive=True), gr.update(interactive=True), ""

    controls = [submit_btn, user_input, status_output]
    for trigger in (submit_btn.click, user_input.submit):   # button click, and the Enter key
        trigger(lock_ui, None, controls, queue=False, trigger_mode="once") \
            .then(cooking_assistant, user_input, outputs) \
            .then(unlock_ui, None, controls, queue=False)   # .then runs even if the request failed

demo.launch(share=True, debug=True)


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://0b831af8180553e868.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


[debug] Prompt Guard raw output: '0.00043565587839111686'
[debug] Relevance check answer: 'YES'
[debug] Image prompt: Crispy Southern fried chicken pieces with a glossy, seasoned crust, resting on a white ceramic plate with a few lemon wedges and fresh parsley garnish, a side of buttery biscuits on the plate, set on a light wooden table, bright natural daylight, shot from a 45-degree angle, professional food photography, 85mm lens, shallow depth of field, bright natural daylight, appetizing, vibrant colors, high detail, sharp focus, no text, no watermark
[debug] Prompt Guard raw output: '0.00043565587839111686'
[debug] Relevance check answer: 'YES'
[debug] Image prompt: A buttermilk fried chicken sandwich showing a thick, golden crusted chicken thigh spilling over the edge of a toasted brioche bun, topped with bright green lettuce, thinly sliced pink pickles, and a dollop of creamy mayo, placed on a light-colored ceramic plate with a small side of curly fries, on a wooden table, bright

## 7. Prompt experiments

 runs the same request at several temperatures and prints the dishes chosen. It makes one call per temperature (taking in consideration to respect the 8K tokens/minute limit).

 Also, we can do experiments with changing the max_tokens permitted limit.

In [11]:
def run_prompt_experiment(request="Create a creative, light Mediterranean-Asian fusion dinner for four, ready in 35 minutes or less. Use affordable Hungarian supermarket ingredients, include protein and fresh vegetables, and avoid deep-frying. ", temperatures=(0.2, 0.8, 1.2), runs=1):
    """Runs the same request several times per temperature to compare variety and validity."""
    results = {}
    for t in temperatures:
        dishes = []
        for _ in range(runs):
            try:
                response = client.chat.completions.create(
                    model=MAIN_MODEL,
                    messages=[
                        {"role": "system", "content": SYSTEM_PROMPT},
                        {"role": "user", "content": f"User request: {request}"},
                    ],
                    temperature=t,
                    top_p=MAIN_TOP_P,
                    max_tokens=MAIN_MAX_TOKENS,
                    reasoning_effort=MAIN_REASONING_EFFORT,
                    response_format={"type": "json_object"},
                )
                recipe = parse_json_response(response.choices[0].message.content)
                validate_recipe(recipe)
                dishes.append(recipe["dish_name"])
            except Exception as e:
                dishes.append(f"[failed: {str(friendly_error(e))[:50]}]")
            time.sleep(30)  # gpt-oss-120b allows only 8K tokens/minute on the free tier
        results[t] = dishes
        print(f"temperature={t}: {dishes}")
    return results


run_prompt_experiment(runs=3);


temperature=0.2: ['Lemon‑Herb Grilled Chicken with Quinoa‑Tabbouleh & Soy‑Sesame Dressing', 'Lemon‑Herb Chicken with Quinoa‑Tabbouleh & Sesame‑Soy Dressing', 'Lemon‑Herb Grilled Chicken with Sesame‑Soy Quinoa Tabbouleh']
temperature=0.8: ['Lemon‑Herb Grilled Chicken with Quinoa‑Tabbouleh & Soy‑Sesame Dressing', 'Citrus‑Herb Chicken with Quinoa‑Tabbouleh & Sesame‑Soy Dressing', 'Herb‑Grilled Chicken with Quinoa‑Tabbouleh & Soy‑Ginger Vinaigrette']
temperature=1.2: ['Citrus‑Sesame Baked Cod over Herbed Quinoa with Asian‑Mediterranean Veggie Medley', 'Lemon‑Soy Chicken Quinoa Bowl', 'Sesame‑Lime Shrimp & Quinoa Veggie Bowl']


Increasing the temperature generally made the dish recommendations more varied and encouraged different ingredient combinations.

## 8. Project documentation

**Idea.** A chatbot that turns a vague craving into a full cooking package (dish, reasoning, ingredients, steps, history, example reviews, image). It goes beyond Q&A: the output is a multi-part, structured, multimodal result produced by a multi-step pipeline.

**Pipeline and models**

| Step | What happens | Model / tool |
|---|---|---|
| 0 | Empty / too-long input rejected | plain Python |
| 1 | Prompt-injection / jailbreak detection | Llama Prompt Guard 2 (Groq) |
| 2 | "Is this about food?" YES/NO | GPT-OSS 20B (Groq), few-shot, low reasoning |
| 3 | Recipe package as JSON | GPT-OSS 120B (Groq) |
| 4 | Validation, average rating, markdown formatting | plain Python |
| 5 | Dish image (placeholder card if unavailable) | Pollinations (keyless, ~1 image / 15 s) + Pillow |
| 6 | Web UI | Gradio |

**Prompt techniques:** persona (Chef Aria), few-shot (full JSON example), chain-of-thought (`reasoning` field first), few-shot classification (topic check), structured/constrained output (strict JSON + rules), model-generated image prompt, and code-controlled variation (random target rating).

**Parameters:** `temperature` 0.8 (variety) / 0 for the classifier (determinism) / 0.4 on retry; `top_p` 0.95; `max_tokens` 2500 (reasoning tokens count too, but the free tier allows only 8K tokens/minute on gpt-oss-120b); `reasoning_effort` low; client `timeout` 45 s and `max_retries` 2.

**Error handling:** the submit button and text box are locked while a request runs (no duplicate requests, a "please wait" sentence is shown); empty and over-long input; API timeout, connection error, rate limit and bad-key messages; both checks fail open; malformed JSON is parsed tolerantly, validated and retried at a lower temperature; ratings clamped to 1-5; image failure falls back gracefully; a top-level try/except keeps the UI alive.

**Design decisions and limitations**
- Prompt Guard only detects malicious prompts, so topic relevance is a separate, deliberate check.
- The history text and reviews are AI-generated and labelled as such in the UI; the history may contain inaccuracies and the reviews are not from real people.
- The image comes from Pollinations, a free third-party service that allows about one keyless image per 15 seconds and may be slow, throttle, or change; the app works without the image.
- Free-tier limits on my account for `gpt-oss-120b`: 30 requests/min, 8K tokens/min, 1K requests/day. Prompt Guard and `gpt-oss-20b` have separate limits.
- Groq offers no image-generation models, so the image comes from a separate service.